# Ablation A — MLP-LoRA E7 (Targets FFN Layers)

**Hypothesis:** Adapting only attention layers (q/k/v/o) leaves no capacity for factual storage,
because facts are stored in FFN/MLP layers (Geva et al. 2021, Dai et al. 2022, Meng et al. 2022).
By adding `gate_proj, up_proj, down_proj` and bumping rank to 32, we give the student
dedicated parameters to absorb teacher's clinical knowledge.

**Method:** E7 Decision-Only (best single-objective Phase 1 method)
**Sizes:** 1.5B + 3B (skip 7B for time)
**Training:** Same 6000 train samples, 2 epochs, decision-only loss
**Output:** `runs/e7_mlp_lora/{model}/`

⚠️ Restart kernel between models on Windows.

In [ ]:
# Cell 0: Environment + model selection
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

import torch
torch.cuda.set_per_process_memory_fraction(0.80)

TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"

use_bf16 = True

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Will train: {TRAIN_MODEL} | bf16={use_bf16}")
print("⚠️ Restart kernel between models")

In [ ]:
# Cell 1: Imports + data
import os, sys, json, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,
)
from peft import LoraConfig, get_peft_model

sys.path.insert(0, os.path.expanduser("~/kd_project"))
sys.path.insert(0, r"C:\Users\adishalit1\Desktop\kd_project")
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, LR, SEED,
    STUDENTS, load_data,
    find_decision_span_chars,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "e1_mlp_lora")
os.makedirs(OUT_DIR, exist_ok=True)

prompts, teacher_rows = load_data()
print(f"Output dir: {OUT_DIR}")

In [ ]:
# Cell 2: Dataset (E7 decision-only style)
class DecOnlyDataset(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct):
        print("  Precomputing dataset...")
        self.items = []
        for r in tqdm(rows, desc="  Tokenizing"):
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(
                tokenizer, prompt_text, answer
            )

            # Decision mask — same as E7
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + ds or s >= answer_start + de):
                        dec_mask[i] = True

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "dec_mask": dec_mask,
            })
        print(f"  ✅ {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

class DecOnlyTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        dec_mask = inputs.pop("dec_mask")
        outputs = model(**inputs)
        sl = outputs.logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        V = sl.size(-1)
        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()
        da = dm.float() * active
        loss = (tok_loss * da).sum() / da.sum().clamp(min=1.0)
        return (loss, outputs) if return_outputs else loss

print("✅ Dataset + trainer ready")

In [ ]:
# Cell 3: Train with MLP-LoRA
cfg = STUDENTS[TRAIN_MODEL]
out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

if os.path.exists(os.path.join(out_path, "adapter_config.json")):
    print(f"⏩ {TRAIN_MODEL} MLP-LoRA already trained")
else:
    print(f"\nLoading {TRAIN_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg["path"], torch_dtype=torch.bfloat16,
        device_map="auto", trust_remote_code=True)

    # ── KEY DIFFERENCE: target FFN layers + higher rank ──
    lora_cfg = LoraConfig(
        r=32,                # was 16 in original E7
        lora_alpha=64,       # was 32
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",   # attention (same as E7)
            "gate_proj", "up_proj", "down_proj",      # ✨ NEW: MLP/FFN layers
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.train()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  ✅ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    print(f"  ✅ ~{trainable/4_358_144:.1f}x more trainable params than original E7 (r=16, attn-only)")

    dataset = DecOnlyDataset(teacher_rows, prompts, tokenizer, cfg["is_instruct"])
    collator = FlexibleCollator(tokenizer, extra_1d_fields=["dec_mask"])

    trainer = DecOnlyTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_path,
            num_train_epochs=2,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=32,
            learning_rate=LR,
            logging_steps=25,
            save_strategy="no",
            bf16=use_bf16,
            seed=SEED,
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=0,
            label_smoothing_factor=0.1,
        ),
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"\n✅ Saved → {out_path}")